<a href="https://colab.research.google.com/github/young-tryler/ThucHanhDeepLearning/blob/main/2001230980_BuiQuocTri_B4_THDL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 2001230980 - Bùi Quốc Trí  - Lab4

In [ ]:
# LAB 4 - AUTOENCODER
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import fashion_mnist

# ==========================================
# 1. NẠP VÀ TIỀN XỬ LÝ DỮ LIỆU
# ==========================================
# Tải tập dữ liệu ảnh thời trang Fashion MNIST
(X_train_raw, _), (X_test_raw, _) = fashion_mnist.load_data()

# Chuẩn hóa giá trị điểm ảnh về đoạn [0, 1] kiểu float32
X_train = X_train_raw.astype('float32') / 255.0
X_test = X_test_raw.astype('float32') / 255.0

# Duỗi phẳng ảnh từ ma trận 2D (28x28) thành Vector 1D (784) phù hợp mạng ANN phẳng
X_train = X_train.reshape((len(X_train), np.prod(X_train.shape[1:])))
X_test = X_test.reshape((len(X_test), np.prod(X_test.shape[1:])))

print("Kích thước dữ liệu Train sau duỗi phẳng:", X_train.shape) # (60000, 784)
print("Kích thước dữ liệu Test sau duỗi phẳng:", X_test.shape)   # (10000, 784)

# ==========================================
# 2. ĐỊNH NGHĨA KIẾN TRÚC AUTOENCODER
# ==========================================
# Kích thước chiều dữ liệu gốc (28 * 28 = 784)
encoding_dim = 32  # Kích thước không gian ẩn (Nén giảm chiều dữ liệu xuống còn 32 số)

# --- Khởi tạo tầng Input ---
input_img = layers.Input(shape=(784,))

# --- Thiết lập mạng ENCODER ---
# Tầng Dense nén dữ liệu từ 784 về 32 kích hoạt bởi hàm relu
encoded = layers.Dense(encoding_dim, activation='relu')(input_img)

# --- Thiết lập mạng DECODER ---
# Tầng Dense giải nén từ 32 quay ngược về 784, kích hoạt sigmoid đưa giá trị về khoảng [0, 1]
decoded = layers.Dense(784, activation='sigmoid')(encoded)

# --- Tạo Model Autoencoder tổng kết hợp ---
autoencoder = models.Model(input_img, decoded)

# --- Tách riêng Model Encoder độc lập (Để lấy vector nén khi cần) ---
encoder = models.Model(input_img, encoded)

# --- Tách riêng Model Decoder độc lập ---
encoded_input = layers.Input(shape=(encoding_dim,))
decoder_layer = autoencoder.layers[-1] # Lấy tầng cuối cùng của chuỗi autoencoder
decoder = models.Model(encoded_input, decoder_layer(encoded_input))

print("\n--- Cấu trúc tổng thể mạng Autoencoder ---")
autoencoder.summary()

# ==========================================
# 3. BIÊN DỊCH VÀ HUẤN LUYỆN MÔ HÌNH
# ==========================================
# Sử dụng thuật toán tối ưu Adam và hàm mất mát binary_crossentropy (đo lường độ lệch pixel)
autoencoder.compile(optimizer='adam', loss='binary_crossentropy')

print("\n--- Bắt đầu huấn luyện Autoencoder ---")
# LƯU Ý: Mục tiêu đầu vào và đầu ra đều là X_train (Học tự tái dựng)
autoencoder.fit(X_train, X_train,
                epochs=10, # Chạy demo 10 epochs để nghiệm thu nhanh
                batch_size=256,
                shuffle=True,
                validation_data=(X_test, X_test))

# ==========================================
# 4. TIẾN HÀNH TÁI DỰNG VÀ TRỰC QUAN HÓA KẾT QUẢ
# ==========================================
# Dự báo (giải nén tái dựng) trên tập dữ liệu kiểm thử
encoded_imgs = encoder.predict(X_test)
decoded_imgs = decoder.predict(encoded_imgs)

# Vẽ biểu đồ hiển thị so sánh ảnh gốc và ảnh sau khi tái dựng
n = 10  # Số lượng ảnh mẫu hiển thị
plt.figure(figsize=(20, 4))
for i in range(n):
    # 1. Hiển thị ảnh gốc (Original)
    ax = plt.subplot(2, n, i + 1)
    plt.imshow(X_test[i].reshape(28, 28))
    plt.gray()
    ax.get_xaxis().set_visible(False)
    ax.get_yaxis().set_visible(False)
    if i == 0:
        ax.set_title("Ảnh Gốc")

    # 2. Hiển thị ảnh tái dựng (Reconstructed)
    ax = plt.subplot(2, n, i + 1 + n)
    plt.imshow(decoded_imgs[i].reshape(28, 28))
    plt.gray()
    ax.get_xaxis().set_visible(False)
    ax.get_yaxis().set_visible(False)
    if i == 0:
        ax.set_title("Ảnh Tái Dựng")

plt.show()

In [ ]:
#LAB 4 - RECURRENT NEURAL NETWORK (RNN)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, SimpleRNN

# ==========================================
# 1. KHỞI TẠO DỮ LIỆU MẪU (GIẢ LẬP ĐÚNG THEO DATASET SUNSPOTS TRONG FILE)
# ==========================================
# Vì tài liệu sử dụng tập dữ liệu Sunspots (vết ban mặt trời), ta giả lập một chuỗi thời gian hình sin có nhiễu giống thực tế
np.random.seed(42)
time_points = np.linspace(0, 50, 250)
sunspots_data = np.sin(time_points) + np.random.normal(0, 0.1, len(time_points))

# Đưa về dạng DataFrame
df = pd.DataFrame(sunspots_data, columns=['Sunspots'])

# ==========================================
# 2. TIỀN XỬ LÝ CHUỖI THỜI GIAN & TẠO TIME STEPS
# ==========================================
# Hàm biến đổi chuỗi 1 chiều thành cặp (X, y) với số bước thời gian (time_steps) xác định
def convert_to_matrix(data, step=4):
    X, Y = [], []
    for i in range(len(data) - step):
        X.append(data[i:(i + step)])
        Y.append(data[i + step])
    return np.array(X), np.array(Y)

# Thiết lập số bước thời gian (Mục 2.4 trong tài liệu đặt step = 4)
step = 4
values = df['Sunspots'].values

X_matrix, Y_matrix = convert_to_matrix(values, step)

# Chia tập dữ liệu thành Train và Test (Tài liệu chia khoảng 70% - 80% dữ liệu làm Train)
train_size = int(len(X_matrix) * 0.75)

X_train, X_test = X_matrix[:train_size], X_matrix[train_size:]
trainy, testy = Y_matrix[:train_size], Y_matrix[train_size:]

# BIẾN ĐỔI QUAN TRỌNG: Reshape X sang chuẩn Tensor 3 chiều [samples, time_steps, features]
X_train = np.reshape(X_train, (X_train.shape[0], step, 1))
X_test = np.reshape(X_test, (X_test.shape[0], step, 1))

print("Kích thước cấu trúc ma trận đầu vào RNN:")
print("-> X_train shape:", X_train.shape) # (Số lượng mẫu, 4, 1)
print("-> X_test shape:", X_test.shape)

# ==========================================
# 3. XÂY DỰNG VÀ BIÊN DỊCH KIẾN TRÚC MẠNG RNN
# ==========================================
model = Sequential()

# Tầng 1: SimpleRNN với 32 đơn vị (units), nhận input_shape là (4, 1)
model.add(SimpleRNN(units=32, input_shape=(step, 1), activation="tanh"))

# Tầng 2: Tầng Dense đầu ra trả về 1 giá trị dự báo duy nhất cho thời điểm tiếp theo
model.add(Dense(units=1))

# Biên dịch mô hình với thuật toán tối ưu RMSprop (hoặc Adam) và hàm mất mát Mean Squared Error (MSE)
model.compile(optimizer='rmsprop', loss='mean_squared_error')

print("\n--- Cấu trúc mô hình SimpleRNN ---")
model.summary()

# ==========================================
# 4. HUẤN LUYỆN MÔ HÌNH
# ==========================================
print("\n--- Bắt đầu huấn luyện RNN ---")
# Huấn luyện trong 20 Epochs với batch_size thích hợp
model.fit(X_train, trainy, epochs=20, batch_size=16, verbose=1)

# ==========================================
# 5. DỰ BÁO VÀ ĐÁNH GIÁ KẾT QUẢ (MỤC 2.8 TRONG FILE PDF)
# ==========================================
# Thực hiện dự báo trên cả hai tập để vẽ đồ thị toàn diện
train_predict = model.predict(X_train)
test_predict = model.predict(X_test)

# Hàm vẽ biểu đồ hiển thị so sánh đúng theo cấu trúc trực quan cuối trang của tài liệu
def plot_result(trainy, testy, train_predict, test_predict):
    actual = np.append(trainy, testy)
    predictions = np.append(train_predict, test_predict)
    rows = len(actual)

    plt.figure(figsize=(15, 6), dpi=80)
    plt.plot(range(rows), actual, label="Actual (Thực tế)", color="blue")
    plt.plot(range(rows), predictions, label="Predictions (Dự báo)", color="orange")

    # Vẽ đường thẳng đứng màu đỏ phân tách giữa tập huấn luyện (Train) và tập kiểm thử (Test)
    plt.axvline(x=len(trainy), color='r', linestyle='--', label="Ranh giới Train - Test")

    plt.legend()
    plt.xlabel("Observation number after given time steps")
    plt.ylabel("Sunspots scaled")
    plt.title("Actual and Predicted Values. The Red Line Separates The Training And Test Examples")
    plt.show()

# Gọi hàm hiển thị đồ thị kết quả dự báo chuỗi thời gian
plot_result(trainy, testy, train_predict, test_predict)